In [2]:
# =============================================================================
#  Paso 1: Importar librerías
# =============================================================================
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.metrics import make_scorer, recall_score, cohen_kappa_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# =============================================================================
#  Paso 2: Cargar datos y definir atributos
# =============================================================================

# --- Configuración para Descriptores Moleculares ---
ruta_archivo_csv = r"C:\Users\benja\Desktop\BD PAMPA\Calculos descriptores moleculares\training_dm.csv"
columna_clase = "Actividad"

# Lista con los 5 descriptores moleculares de consenso
atributos_seleccionados = [
    'LOGPcons',
    'piPC05',
    'CATS2D_07_AP',
    'B06[C-C]',
    'Eig12_EA(dm)'
]

# --- Carga y Preparación de Datos ---
try:
    df_completo = pd.read_csv(ruta_archivo_csv)
    df_completo.columns = df_completo.columns.str.strip()
    X = df_completo[atributos_seleccionados]
    y = df_completo[columna_clase]
    print(f"Datos cargados. Se usarán {X.shape[1]} atributos para {X.shape[0]} moléculas.")

    # =============================================================================
    #  Paso 3: Definir las métricas de evaluación
    # =============================================================================
    labels = sorted(y.unique())
    neg_label, pos_label = labels[0], labels[1]
    print(f"Clase Positiva detectada: '{pos_label}', Clase Negativa: '{neg_label}'")

    scoring = {
        'Accuracy': 'accuracy',
        'BACC': 'balanced_accuracy',
        'Sensitivity': make_scorer(recall_score, pos_label=pos_label),
        'Specificity': make_scorer(recall_score, pos_label=neg_label),
        'Kappa': make_scorer(cohen_kappa_score),
        'ROC_AUC': 'roc_auc'
    }

    # =============================================================================
    #  Paso 4: Definir y Evaluar Modelos
    # =============================================================================
    modelos = {
        "Árbol de Decisión (J48)": DecisionTreeClassifier(random_state=42),
        "Regresión Logística": LogisticRegression(max_iter=1000, random_state=42),
        "k-NN (IBk, k=5)": KNeighborsClassifier(n_neighbors=5),
        "Random Forest": RandomForestClassifier(random_state=42),
        "SVM": SVC(probability=True, random_state=42)
    }
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    resultados = {}

    print("\n--- Iniciando Evaluación de Modelos (Descriptores Moleculares) ---")
    for nombre, modelo in modelos.items():
        scores = cross_validate(modelo, X, y, cv=cv, scoring=scoring)
        resultados[nombre] = {key: np.mean(values) for key, values in scores.items()}
        print(f"  > Evaluado: {nombre}")
    print("--- Evaluación Completada ---\n")

    # =============================================================================
    #  Paso 5: Mostrar y Guardar resultados
    # =============================================================================
    df_resultados = pd.DataFrame.from_dict(resultados, orient='index')
    df_resultados = df_resultados.rename(columns={
        'test_Accuracy': 'Accuracy', 'test_BACC': 'BACC',
        'test_Sensitivity': 'Sensitivity', 'test_Specificity': 'Specificity',
        'test_Kappa': 'Kappa', 'test_ROC_AUC': 'ROC AUC'
    })
    columnas_ordenadas = ['Accuracy', 'BACC', 'Sensitivity', 'Specificity', 'Kappa', 'ROC AUC']
    df_resultados = df_resultados[columnas_ordenadas]

    print("📊 Tabla Comparativa de Rendimiento (Descriptores Moleculares):")
    print(df_resultados.round(3))

    nombre_archivo_salida = 'resultados_completos_dm.csv'
    df_resultados.to_csv(nombre_archivo_salida)

    print(f"\n✅ ¡Tabla de resultados guardada exitosamente en el archivo '{nombre_archivo_salida}'!")

except Exception as e:
    print(f"❌ Ocurrió un error inesperado durante la ejecución: {e}")

Datos cargados. Se usarán 5 atributos para 4357 moléculas.
Clase Positiva detectada: 'Act1', Clase Negativa: 'Act-1'

--- Iniciando Evaluación de Modelos (Descriptores Moleculares) ---
  > Evaluado: Árbol de Decisión (J48)
  > Evaluado: Regresión Logística
  > Evaluado: k-NN (IBk, k=5)
  > Evaluado: Random Forest
  > Evaluado: SVM
--- Evaluación Completada ---

📊 Tabla Comparativa de Rendimiento (Descriptores Moleculares):
                         Accuracy   BACC  Sensitivity  Specificity  Kappa  \
Árbol de Decisión (J48)     0.653  0.653        0.663        0.642  0.305   
Regresión Logística         0.733  0.725        0.841        0.608  0.455   
k-NN (IBk, k=5)             0.713  0.709        0.774        0.643  0.420   
Random Forest               0.715  0.711        0.763        0.660  0.424   
SVM                         0.738  0.728        0.863        0.594  0.464   

                         ROC AUC  
Árbol de Decisión (J48)    0.657  
Regresión Logística        0.765  
k-NN 

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.svm import SVC

# =============================================================================
#  Paso 1: Cargar tu CONJUNTO DE ENTRENAMIENTO
# =============================================================================

# --- Configuración para Descriptores Moleculares ---
# Este es tu archivo de 4258 moléculas
ruta_archivo_csv = r"C:\Users\benja\Desktop\BD PAMPA\Calculos descriptores moleculares\training_dm.csv"
columna_clase = "Actividad"
atributos_seleccionados = [
    'LOGPcons',
    'piPC05',
    'CATS2D_07_AP',
    'B06[C-C]',
    'Eig12_EA(dm)'
]

# --- Carga y Preparación de Datos ---
try:
    df_completo = pd.read_csv(ruta_archivo_csv)
    X = df_completo[atributos_seleccionados]
    y = df_completo[columna_clase]
    print(f"Datos de ENTRENAMIENTO cargados para la optimización: {X.shape[0]} moléculas.")

    # =============================================================================
    #  Paso 2: Definir el "Grid" de Hiperparámetros a Probar
    # =============================================================================
    param_grid = {
        'C': [0.1, 1, 10, 100, 1000],
        'gamma': [1, 0.1, 0.01, 0.001, 'scale'],
        'kernel': ['rbf']
    }

    # =============================================================================
    #  Paso 3: Configurar y Ejecutar GridSearchCV
    # =============================================================================
    print("\n--- Iniciando búsqueda de hiperparámetros (puede tardar varios minutos)... ---")
    
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    grid_search = GridSearchCV(SVC(probability=True, random_state=42), 
                               param_grid, 
                               cv=cv, 
                               scoring='roc_auc', 
                               n_jobs=-1,
                               verbose=1) # Añadido para que veas el progreso
    grid_search.fit(X, y)

    print("--- ¡Búsqueda completada! ---")

    # =============================================================================
    #  Paso 4: Mostrar los resultados de la optimización
    # =============================================================================
    print("\n🏆 Mejores Hiperparámetros Encontrados:")
    print(grid_search.best_params_)

    print(f"\n📈 Mejor puntuación ROC AUC (promedio en validación cruzada sobre el set de entrenamiento): {grid_search.best_score_:.4f}")
    
    # Recordatorio del resultado anterior sin tunear
    print(f"(Resultado anterior sin tunear: 0.793)")

except FileNotFoundError:
    print(f"❌ ERROR: No se encontró el archivo '{ruta_archivo_csv}'.")
    print("   Asegúrate de que el nombre del archivo sea correcto y esté en la misma carpeta.")
except Exception as e:
    print(f"❌ Ocurrió un error inesperado durante la ejecución: {e}")

Datos de ENTRENAMIENTO cargados para la optimización: 4357 moléculas.

--- Iniciando búsqueda de hiperparámetros (puede tardar varios minutos)... ---
Fitting 10 folds for each of 25 candidates, totalling 250 fits
--- ¡Búsqueda completada! ---

🏆 Mejores Hiperparámetros Encontrados:
{'C': 100, 'gamma': 0.01, 'kernel': 'rbf'}

📈 Mejor puntuación ROC AUC (promedio en validación cruzada sobre el set de entrenamiento): 0.7938
(Resultado anterior sin tunear: 0.793)
